[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/01-python-basics/notebooks/04-mini-project-grade-calculator.ipynb)

# 01 · Notebook 4 — Mini-project: grade calculator (+ bonus text adventure)

**Goal:** combine everything from this module into a small, useful program.

**The brief:** a teacher has a CSV file (**CSV** = comma-separated values, a simple text table) of marks. Some rows are messy. Build a program that:
1. reads the file,
2. skips bad rows with a clear message instead of crashing,
3. computes each student's average and letter grade,
4. prints a neat ranked report and class statistics,
5. saves the results to JSON.

We build it in small functions and test each one before moving on. That is how professionals work.

## Step 1 — Create the data file

In a real project the file would already exist. Here we write it ourselves so the notebook works anywhere (including Colab). Two rows are broken on purpose: `abc` is not a number, and `101` is out of range.

In [1]:
from pathlib import Path

raw = """name,maths,science,english
Asha,88,92,79
Ravi,95,89,91
Meera,72,abc,80
John,55,61,48
Priya,101,90,85
Karan,66,74,70
Fatima,91,94,97
"""
csv_path = Path("marks.csv")
csv_path.write_text(raw, encoding="utf-8")
print(csv_path.read_text())

name,maths,science,english
Asha,88,92,79
Ravi,95,89,91
Meera,72,abc,80
John,55,61,48
Priya,101,90,85
Karan,66,74,70
Fatima,91,94,97



## Step 2 — Read the CSV into dicts

The standard library's `csv.DictReader` turns each row into a dict using the header line as keys. That means we can write `row["maths"]` instead of remembering column positions.

In [2]:
import csv

with open(csv_path, encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    subjects = [col for col in reader.fieldnames if col != "name"]

print("Subjects:", subjects)
print("First row:", rows[0])
print("Number of rows:", len(rows))

Subjects: ['maths', 'science', 'english']
First row: {'name': 'Asha', 'maths': '88', 'science': '92', 'english': '79'}
Number of rows: 7


## Step 3 — The grade function

We store the grade boundaries as a list of tuples, highest first, and return the first one the score reaches. Storing rules as data (instead of a long `if` chain) makes them easy to change. We test it right away.

In [3]:
GRADE_BOUNDARIES = [(90, "A"), (75, "B"), (60, "C"), (40, "D"), (0, "F")]

def letter_grade(score):
    for lowest, letter in GRADE_BOUNDARIES:
        if score >= lowest:
            return letter
    raise ValueError(f"score cannot be negative: {score}")

for test in [95, 90, 89.9, 75, 60, 45, 10]:
    print(test, "->", letter_grade(test))

95 -> A
90 -> A
89.9 -> B
75 -> B
60 -> C
45 -> D
10 -> F


## Step 4 — Parse one row safely

`parse_marks` converts the text marks to numbers. It **raises** `ValueError` for anything wrong, with a message saying what. We test it on a good row and a bad row.

In [4]:
def parse_marks(row, subjects):
    marks = []
    for subject in subjects:
        mark = float(row[subject].strip())          # ValueError if not a number
        if not 0 <= mark <= 100:
            raise ValueError(f"{subject} mark {mark} is outside 0-100")
        marks.append(mark)
    return marks

print(parse_marks(rows[0], subjects))
try:
    parse_marks(rows[2], subjects)
except ValueError as err:
    print("Bad row:", err)

[88.0, 92.0, 79.0]
Bad row: could not convert string to float: 'abc'


## Step 5 — Build the report

Loop over the rows. For each, try to parse; on failure, record why and `continue`. On success, store a small dict of results. Finally sort by average, best first, using a `lambda` as the sort key.

In [5]:
def build_report(rows, subjects):
    results, skipped = [], []
    for line_number, row in enumerate(rows, start=2):   # line 1 is the header
        try:
            marks = parse_marks(row, subjects)
        except ValueError as err:
            skipped.append(f"line {line_number} ({row['name']}): {err}")
            continue
        avg = sum(marks) / len(marks)
        results.append({
            "name": row["name"],
            "marks": dict(zip(subjects, marks)),
            "average": round(avg, 1),
            "grade": letter_grade(avg),
        })
    results.sort(key=lambda r: r["average"], reverse=True)
    return results, skipped

results, skipped = build_report(rows, subjects)
print(len(results), "students graded,", len(skipped), "skipped")

5 students graded, 2 skipped


## Step 6 — Print a neat table

f-string alignment (`<10` left, `>8.1f` right with 1 decimal) lines the columns up.

In [6]:
def print_report(results, skipped):
    print(f"{'Rank':<5}{'Name':<10}{'Average':>8}  Grade")
    print("-" * 31)
    for rank, r in enumerate(results, start=1):
        print(f"{rank:<5}{r['name']:<10}{r['average']:>8.1f}  {r['grade']}")
    for message in skipped:
        print("Skipped", message)

print_report(results, skipped)

Rank Name       Average  Grade
-------------------------------
1    Fatima        94.0  A
2    Ravi          91.7  A
3    Asha          86.3  B
4    Karan         70.0  C
5    John          54.7  D
Skipped line 4 (Meera): could not convert string to float: 'abc'
Skipped line 6 (Priya): maths mark 101.0 is outside 0-100


## Step 7 — Class statistics

Comprehensions and dicts answer questions about the class: average per subject, the grade distribution, and the top student per subject.

In [7]:
subject_averages = {
    s: round(sum(r["marks"][s] for r in results) / len(results), 1) for s in subjects
}
print("Subject averages:", subject_averages)

grade_counts = {}
for r in results:
    grade_counts[r["grade"]] = grade_counts.get(r["grade"], 0) + 1
print("Grade counts:", dict(sorted(grade_counts.items())))

for s in subjects:
    best = max(results, key=lambda r: r["marks"][s])
    print(f"Top in {s}: {best['name']} ({best['marks'][s]:.0f})")

Subject averages: {'maths': 79.0, 'science': 82.0, 'english': 77.0}
Grade counts: {'A': 2, 'B': 1, 'C': 1, 'D': 1}
Top in maths: Ravi (95)
Top in science: Fatima (94)
Top in english: Fatima (97)


## Step 8 — Save the results as JSON

Other programs (a web page, a spreadsheet, an AI agent) can read this file. We read it back to prove it worked.

In [8]:
import json

out_path = Path("grade_report.json")
out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
reloaded = json.loads(out_path.read_text(encoding="utf-8"))
print("Saved and reloaded", len(reloaded), "records. First:", reloaded[0]["name"], reloaded[0]["grade"])

Saved and reloaded 5 records. First: Fatima A


## Step 9 — Wrap it in a class (optional)

The same logic packaged as a `Gradebook` object. This shows how classes bundle data (the rows) with actions (report, stats). Notice it reuses our functions.

In [9]:
class Gradebook:
    def __init__(self, csv_file):
        with open(csv_file, encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            self.rows = list(reader)
            self.subjects = [c for c in reader.fieldnames if c != "name"]
        self.results, self.skipped = build_report(self.rows, self.subjects)

    def top(self, n=3):
        return [r["name"] for r in self.results[:n]]

    def lookup(self, name):
        for r in self.results:
            if r["name"].lower() == name.lower():
                return r
        return None

book = Gradebook(csv_path)
print("Top 3:", book.top())
print("Ravi:", book.lookup("ravi"))
print("Meera:", book.lookup("Meera"))   # None - her row was skipped

Top 3: ['Fatima', 'Ravi', 'Asha']
Ravi: {'name': 'Ravi', 'marks': {'maths': 95.0, 'science': 89.0, 'english': 91.0}, 'average': 91.7, 'grade': 'A'}
Meera: None


## Bonus — A tiny text adventure

Games are a fun way to practise dicts, loops and functions. The whole map is a dict of rooms. Normally the player types commands with `input()`, but notebooks run automatically here, so we feed a **scripted list of commands** instead. To play for real, run `python code/text_adventure.py` in a terminal.

In [10]:
ROOMS = {
    "lab": {"text": "The AI lab. Doors north and east.",
            "exits": {"north": "library", "east": "server room"}},
    "library": {"text": "Books everywhere. A note: 'the key is in the server room'.",
                "exits": {"south": "lab"}},
    "server room": {"text": "Blinking lights. A KEY lies here. Exits: west, locked door north.",
                    "exits": {"west": "lab", "north": "exit"}, "item": "key"},
    "exit": {"text": "You unlock the door and step outside. You win!", "exits": {}},
}

def play(commands):
    room, inventory = "lab", set()
    print(f"[{room}] {ROOMS[room]['text']}")
    for command in commands:
        print(f"> {command}")
        if command == "take":
            item = ROOMS[room].get("item")
            if item:
                inventory.add(item)
                print(f"You take the {item}.")
            else:
                print("Nothing here.")
            continue
        exits = ROOMS[room]["exits"]
        if command not in exits:
            print(f"Can't go {command}. Exits: {', '.join(exits)}")
            continue
        if exits[command] == "exit" and "key" not in inventory:
            print("Locked. You need a key.")
            continue
        room = exits[command]
        print(f"[{room}] {ROOMS[room]['text']}")
        if room == "exit":
            return True
    return False

won = play(["north", "south", "east", "north", "take", "jump", "north"])
print("Won:", won)

[lab] The AI lab. Doors north and east.
> north
[library] Books everywhere. A note: 'the key is in the server room'.
> south
[lab] The AI lab. Doors north and east.
> east
[server room] Blinking lights. A KEY lies here. Exits: west, locked door north.
> north
Locked. You need a key.
> take
You take the key.
> jump
Can't go jump. Exits: west, north
> north
[exit] You unlock the door and step outside. You win!
Won: True


## Clean up

In [11]:
for p in [csv_path, out_path]:
    p.unlink(missing_ok=True)
print("Removed practice files.")

Removed practice files.


## Extend it yourself

1. Add a `history` subject column to the CSV. Does your code need changes? (It should not — that is why we read subjects from the header.)
2. Add **weights**: maths counts double. Compute a weighted average.
3. Write the report to a new CSV with `csv.DictWriter`.
4. In the adventure, add a new room and a second item needed to win.
5. Move the grade functions into `grades.py` and import them in a fresh notebook.

You have used: variables, strings, f-strings, lists, tuples, dicts, sets, comprehensions, conditions, loops, functions, lambdas, exceptions, files, JSON, modules and a class. That is the whole module — well done.